# 01 - Data Inventory

**Goal:** Catalog all files from the course before processing.

**What we do:**
1. Recursively scan `data/raw/`
2. Inspect every file (PDF / Excel / docx / txt) and extract metadata
3. Assign a chunking strategy to each file
4. Save `data/processed/inventory.json` for downstream notebooks

**Key design decision:**
> ALL files - PDF, Excel, docx - go into **FAISS**.
> Every file contains course knowledge used for RAG, individual program generation, and AI coaching.
> No PostgreSQL at this stage.

**After this notebook we know:**
- How many files of each type we have
- Estimated token count and expected number of chunks
- Which files need special handling (tables, very long, very short)
- The chunking strategy for each file

## Setup

In [ ]:
import json
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings

warnings.filterwarnings('ignore')

import fitz  
import openpyxl 
import docx 
import pandas as pd

# Paths
NOTEBOOK_DIR = Path().resolve()      
BACKEND_DIR = NOTEBOOK_DIR.parent 
RAW_DIR = BACKEND_DIR / 'data' / 'raw'
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# All supported file types
SUPPORTED_EXTENSIONS = {'.pdf', '.xlsx', '.xls', '.docx', '.doc', '.txt'}

# Rough token estimate multipliers
WORDS_TO_TOKENS = 1.3   # for text-based files
EXCEL_TOKENS_ROW = 25    # average tokens per Excel row
CHUNK_TARGET_TOKENS = 400  # for estimating chunk count

print('Setup complete')
print(f'  RAW_DIR: {RAW_DIR}')
print(f'  PROCESSED_DIR: {PROCESSED_DIR}')
print(f'  RAW exists: {RAW_DIR.exists()}')

Setup complete
  RAW_DIR: D:\Cybernetic Gym Assistant\backend\data\raw
  PROCESSED_DIR: D:\Cybernetic Gym Assistant\backend\data\processed
  RAW exists: True


## 1. Scan data/raw/

In [4]:
def scan_directory(root: Path) -> list[dict]:
    """Recursively scan a directory and return file metadata for all supported types."""
    files = []
    if not root.exists():
        print(f'WARNING: {root} does not exist. Add course files there and re-run.')
        return files

    for path in sorted(root.rglob('*')):
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
            rel = path.relative_to(root)
            size = path.stat().st_size
            files.append({
                'filename': path.name,
                'ext': path.suffix.lower(),
                'rel_path': str(rel),
                'abs_path': str(path),
                'size_kb': round(size / 1024, 1),
                'folder': str(rel.parent),
            })
    return files


raw_files = scan_directory(RAW_DIR)
ext_counts = Counter(f['ext'] for f in raw_files)

print(f'Found {len(raw_files)} files in {RAW_DIR}\n')
for ext, n in sorted(ext_counts.items()):
    print(f'{ext:8s} -> {n} file(s)')

Found 88 files in D:\Cybernetic Gym Assistant\backend\data\raw

.docx    -> 3 file(s)
.pdf     -> 70 file(s)
.xlsx    -> 15 file(s)


## 2. File Inspectors

In [6]:
def inspect_pdf(path: str) -> dict:
    """Extract metadata from a PDF file using PyMuPDF."""
    try:
        doc = fitz.open(path)
        num_pages = doc.page_count
        total_words = 0
        has_tables = False
        preview = ''

        for page in doc:
            text = page.get_text()
            total_words += len(text.split())
            if not preview and text.strip():
                preview = text[:500].replace('\n', ' ').strip()
            if page.find_tables().tables:
                has_tables = True

        doc.close()
        return {
            'file_type': 'pdf',
            'num_pages': num_pages,
            'total_words': total_words,
            'est_tokens': int(total_words * WORDS_TO_TOKENS),
            'has_tables': has_tables,
            'preview': preview,
            'error': None,
        }
    except Exception as e:
        return {'file_type': 'pdf', 'error': str(e)}


def inspect_excel(path: str) -> dict:
    """
    Extract metadata from an Excel file.
    Excel files contain course knowledge (exercise tables, programming examples,
    reference data) — they go into FAISS like all other files.
    """
    try:
        wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
        sheets = []
        total_rows = 0
        preview = ''

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            rows = ws.max_row or 0
            cols = ws.max_column or 0
            total_rows += rows
            sheets.append({'sheet': sheet_name, 'rows': rows, 'cols': cols})

            # Preview: first 3 rows of the first sheet
            if not preview:
                sample = []
                for i, row in enumerate(ws.iter_rows(values_only=True)):
                    if i >= 3:
                        break
                    sample.append([str(c) if c is not None else '' for c in row[:7]])
                preview = str(sample)

        wb.close()
        return {
            'file_type': 'excel',
            'sheets': sheets,
            'total_rows': total_rows,
            'est_tokens': total_rows * EXCEL_TOKENS_ROW,
            'preview': preview[:400],
            'error': None,
        }
    except Exception as e:
        return {'file_type': 'excel', 'error': str(e)}


def inspect_docx(path: str) -> dict:
    """Extract metadata from a .docx file."""
    try:
        document = docx.Document(path)
        paragraphs = [p.text for p in document.paragraphs if p.text.strip()]
        total_words = sum(len(p.split()) for p in paragraphs)
        return {
            'file_type': 'docx',
            'num_paragraphs': len(paragraphs),
            'num_tables': len(document.tables),
            'total_words': total_words,
            'est_tokens': int(total_words * WORDS_TO_TOKENS),
            'preview': ' '.join(paragraphs[:5])[:500],
            'error': None,
        }
    except Exception as e:
        return {'file_type': 'docx', 'error': str(e)}


def inspect_txt(path: str) -> dict:
    """Extract metadata from a plain text file."""
    try:
        text = Path(path).read_text(encoding='utf-8', errors='ignore')
        words = len(text.split())
        return {
            'file_type': 'txt',
            'total_words': words,
            'est_tokens': int(words * WORDS_TO_TOKENS),
            'preview': text[:500].replace('\n', ' '),
            'error': None,
        }
    except Exception as e:
        return {'file_type': 'txt', 'error': str(e)}


INSPECTORS = {
    '.pdf': inspect_pdf,
    '.xlsx': inspect_excel,
    '.xls': inspect_excel,
    '.docx': inspect_docx,
    '.doc': inspect_docx,
    '.txt': inspect_txt,
}

print('Inspectors defined OK')

Inspectors defined OK


## 3. Run Inspection

In [ ]:
inventory = []

print(f'Inspecting {len(raw_files)} files...\n')
print(f'{"Status":<6} {"Filename":<50} {"Size KB":>8}  {"Est. Tokens":>12}')
print('-' * 80)

for f in raw_files:
    inspector = INSPECTORS.get(f['ext'])
    details = inspector(f['abs_path']) if inspector else {'file_type': 'unknown', 'error': 'unsupported type'}
    entry = {**f, **details}
    inventory.append(entry)

    status = 'OK' if not details.get('error') else 'ERROR'
    tokens = details.get('est_tokens') or 0
    print(f'[{status:<4}] {f["filename"]:<50} {f["size_kb"]:>7.1f}  {tokens:>12,}')

total_tokens = sum((e.get('est_tokens') or 0) for e in inventory)
est_chunks = total_tokens // CHUNK_TARGET_TOKENS

print('-' * 80)
print(f'Total files: {len(inventory)}')
print(f'Total tokens: ~{total_tokens:,}')
print(f'Estimated chunks: ~{est_chunks:,}  (at ~{CHUNK_TARGET_TOKENS} tokens/chunk)')

Inspecting 88 files...

Status Filename                                            Size KB   Est. Tokens
--------------------------------------------------------------------------------
[OK  ] 1RM Calculator MennoHenselmans.com.xlsx              103.5           500
[OK  ] AAS PTC 2022.pdf                                    2558.4        10,397
[OK  ] Ad libitum dieting case study.pdf                    157.4           868
[OK  ] Ad libitum dieting PTC 2022.pdf                     3044.2        20,640
[OK  ] Adherence PTC 2022.pdf                              1927.4        40,270
[OK  ] Advanced strength training techniques PTC 2023.pdf  2652.6        17,121
[OK  ] Age specific programming PTC 2022.pdf               1584.7         5,112
[OK  ] Biochemistry 101 PTC 2022.pdf                        836.2         1,742
[OK  ] Body fat percentage visual reference guide PTC.pdf 10454.8         1,734
[OK  ] Caliper usage case study.xlsx                         76.6        24,500
[OK  ] Carbohy

## 4. Chunking Strategy Assignment

All files go to **FAISS**. Strategy depends on file type and size:

| Condition | Strategy |
|---|---|
| Excel (.xlsx/.xls) | Convert sheets to markdown tables → agentic chunking |
| PDF with tables | PyMuPDF text + table extraction → agentic chunking |
| Very short (<500 words) | Single chunk — no splitting needed |
| Very long (>50K tokens) | Recursive section-aware chunking |
| Everything else | **Agentic chunking via Gemini** (headline + summary + original text) |

In [ ]:
def assign_strategy(e: dict) -> dict:
    """Assign chunking strategy and destination (always FAISS) to a file entry."""
    ft = e.get('file_type', '')
    tokens = e.get('est_tokens') or 0
    words = e.get('total_words') or 0

    if e.get('error'):
        return {'destination': 'SKIP', 'strategy': 'skip — read error'}

    if ft == 'excel':
        return {
            'destination': 'FAISS',
            'strategy': 'excel-to-markdown -> agentic chunking via Gemini',
        }

    if ft == 'pdf' and e.get('has_tables'):
        return {
            'destination': 'FAISS',
            'strategy': 'PyMuPDF text + table extraction -> agentic chunking via Gemini',
        }

    if words < 500:
        return {
            'destination': 'FAISS',
            'strategy': 'single chunk (file too short to split)',
        }

    if tokens > 50_000:
        return {
            'destination': 'FAISS',
            'strategy': 'recursive section-aware chunking (file too long for single LLM call)',
        }

    return {
        'destination': 'FAISS',
        'strategy': 'agentic chunking via Gemini (headline + summary + original text)',
    }


for e in inventory:
    e.update(assign_strategy(e))

print(f'{"Filename":<48} {"Strategy"}')
print('-' * 105)
for e in inventory:
    dest = f'[{e["destination"]}]'
    print(f'{e["filename"]:<48} {e["strategy"]}')

# Count by strategy
strat_counts = Counter(e['strategy'] for e in inventory)
print(f'\nStrategy breakdown:')
for strat, n in strat_counts.most_common():
    print(f'{n}x  {strat}')

Filename                                         Strategy
---------------------------------------------------------------------------------------------------------
1RM Calculator MennoHenselmans.com.xlsx          excel-to-markdown -> agentic chunking via Gemini
AAS PTC 2022.pdf                                 PyMuPDF text + table extraction -> agentic chunking via Gemini
Ad libitum dieting case study.pdf                agentic chunking via Gemini (headline + summary + original text)
Ad libitum dieting PTC 2022.pdf                  agentic chunking via Gemini (headline + summary + original text)
Adherence PTC 2022.pdf                           PyMuPDF text + table extraction -> agentic chunking via Gemini
Advanced strength training techniques PTC 2023.pdf agentic chunking via Gemini (headline + summary + original text)
Age specific programming PTC 2022.pdf            agentic chunking via Gemini (headline + summary + original text)
Biochemistry 101 PTC 2022.pdf                    agentic

## 5. Flags - Files Needing Special Attention

In [ ]:
flags = {
    'read_errors': [e['filename'] for e in inventory if e.get('error')],
    'large_files_500kb': [e['filename'] for e in inventory if e['size_kb'] > 500],
    'pdf_with_tables': [e['filename'] for e in inventory
                            if e.get('file_type') == 'pdf' and e.get('has_tables')],
    'excel_files': [e['filename'] for e in inventory if e.get('file_type') == 'excel'],
    'multi_sheet_excel': [e['filename'] for e in inventory
                            if e.get('file_type') == 'excel' and len(e.get('sheets', [])) > 1],
    'very_long_50k_tok': [e['filename'] for e in inventory if (e.get('est_tokens') or 0) > 50_000],
    'very_short_500_words': [e['filename'] for e in inventory
                             if (e.get('total_words') or 0) < 500 and e.get('file_type') != 'excel'],
}

descriptions = {
    'read_errors': 'Files with read errors (cannot be opened)',
    'large_files_500kb': 'Large files >500 KB',
    'pdf_with_tables': 'PDFs with detected tables (extra table extraction needed)',
    'excel_files': 'Excel files (sheets converted to markdown before chunking)',
    'multi_sheet_excel': 'Excel files with multiple sheets (processed per sheet)',
    'very_long_50k_tok': 'Very long files >50K tokens (recursive chunking)',
    'very_short_500_words': 'Very short files <500 words (single chunk)',
}

print('FLAGS — FILES NEEDING SPECIAL HANDLING:\n')
for key, desc in descriptions.items():
    items = flags[key]
    status = f'{len(items)} file(s)' if items else 'none'
    print(f'  {desc}')
    print(f'  -> {status}')
    for item in items:
        print(f' • {item}')
    print()

FLAGS — FILES NEEDING SPECIAL HANDLING:

  Files with read errors (cannot be opened)
  -> none

  Large files >500 KB
  -> 63 file(s)
       • AAS PTC 2022.pdf
       • Ad libitum dieting PTC 2022.pdf
       • Adherence PTC 2022.pdf
       • Advanced strength training techniques PTC 2023.pdf
       • Age specific programming PTC 2022.pdf
       • Biochemistry 101 PTC 2022.pdf
       • Body fat percentage visual reference guide PTC.pdf
       • Carbohydrates PTC 2022.pdf
       • Cardio PTC 2022.pdf
       • Case study average Joe program design.pdf
       • Case study block periodization and mixing different goals.pdf
       • Case study booty building intake form.pdf
       • Case study Chad BJJ.pdf
       • Case study diet design heavyweight BB.xlsx
       • Case study female Powerlifter prep.pdf
       • Case study John untrained ketogenic.pdf
       • Case study M simple calorie cycling.pdf
       • Case study MP competitor offseason shoulder rehab intake form.pdf
       • Case stu

## 7. Summary DataFrame

In [ ]:
rows = []
for e in inventory:
    row = {
        'Filename': e['filename'],
        'Type': e.get('file_type', '?').upper(),
        'Folder': e['folder'],
        'Size KB': e['size_kb'],
        'Est. Tokens': e.get('est_tokens') or 0,
        'Destination': e.get('destination', '?'),
        'Status': 'ERROR: ' + e['error'] if e.get('error') else 'OK',
    }

    ft = e.get('file_type', '')
    if ft == 'pdf':
        row['Details'] = f"{e.get('num_pages', '?')} pages, tables={e.get('has_tables')}"
    elif ft == 'excel':
        row['Details'] = f"{len(e.get('sheets', []))} sheet(s), {e.get('total_rows', 0)} rows"
    elif ft in ('docx', 'doc'):
        row['Details'] = f"{e.get('num_paragraphs', '?')} paragraphs, {e.get('num_tables', 0)} tables"
    else:
        row['Details'] = f"{e.get('total_words', 0):,} words"

    rows.append(row)

df = pd.DataFrame(rows).sort_values(['Type', 'Filename'])

print('INVENTORY SUMMARY')
print(df.to_string(index=False))

print('\nTYPE STATS')
type_stats = df.groupby('Type').agg(
    Count=('Filename', 'count'),
    Total_KB=('Size KB', 'sum'),
    Total_Tokens=('Est. Tokens', 'sum'),
).reset_index()
type_stats['Est. Chunks'] = (type_stats['Total_Tokens'] / CHUNK_TARGET_TOKENS).astype(int)
print(type_stats.to_string(index=False))

INVENTORY SUMMARY
                                                          Filename  Type Folder  Size KB  Est. Tokens Destination Status                  Details
                       Henselmans Coaching Client Intake Form.docx  DOCX      .    143.6          781       FAISS     OK 43 paragraphs, 26 tables
                           Muscle Functional Anatomy PTC 2023.docx  DOCX      .  42993.4        11754       FAISS     OK 289 paragraphs, 0 tables
                                 Program adherence assessment.docx  DOCX      .     21.4          384       FAISS     OK  30 paragraphs, 0 tables
                           1RM Calculator MennoHenselmans.com.xlsx EXCEL      .    103.5          500       FAISS     OK      1 sheet(s), 20 rows
                                     Caliper usage case study.xlsx EXCEL      .     76.6        24500       FAISS     OK     1 sheet(s), 980 rows
            Case study MP competitor offseason shoulder rehab.xlsx EXCEL      .    229.9        25000     

## 8. Save inventory.json

In [ ]:
clean_inventory = []
for e in inventory:
    entry = {k: v for k, v in e.items() if k != 'abs_path'}
    # Sanitize preview text
    if entry.get('preview'):
        entry['preview'] = str(entry['preview']).encode('utf-8', errors='replace').decode('utf-8')
    clean_inventory.append(entry)

output = {
    'generated_at': datetime.now().isoformat(),
    'total_files': len(clean_inventory),
    'total_est_tokens': total_tokens,
    'estimated_total_chunks': est_chunks,
    'ext_counts': dict(ext_counts),
    'flags': flags,
    'files': clean_inventory,
}

out_path = PROCESSED_DIR / 'inventory.json'
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'inventory.json saved -> {out_path}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')
print(f'\nContains:')
print(f'  {len(clean_inventory)} file entries')
print(f'  {len(flags)} flag categories')

inventory.json saved -> D:\Cybernetic Gym Assistant\backend\data\processed\inventory.json
File size: 68.4 KB

Contains:
  88 file entries
  7 flag categories
